# Goldpreisprognose nach CRISP-DM

Dieses Notebook orchestriert ausschließlich Funktionen aus dem installierbaren Paket. Alle Abbildungen werden als PNG gespeichert und anschließend von der Festplatte angezeigt.

In [ ]:
from pathlib import Path
import sys
from IPython.display import Image, display
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from gold_forecasting.config import load_yaml
from gold_forecasting.data_download import download_all
from gold_forecasting.data_preparation import prepare_dataset
from gold_forecasting.experiments import run_holdout, run_future
from gold_forecasting.plotting import plot_history


## 1. Geschäftsverständnis

Ziel ist die Prognose des täglichen Goldpreises in USD je Feinunze. Die Prognosen unterstützen eine methodische Bewertung, sind aber keine Anlageberatung. Finanzmärkte reagieren auf Regimewechsel, Liquidität und unerwartete Ereignisse; deshalb bleiben selbst gute historische Ergebnisse unsicher. Ein Mehrwert gegenüber der naiven Persistenz ist entscheidend, weil komplexe Modelle sonst keinen belegbaren Zusatznutzen liefern.

## 2. Datenverständnis

Yahoo Finance liefert Gold-Futures (`GC=F`) sowie optionale Reihen für US-Dollar-Index, Silber, Öl, S&P 500, VIX und Bitcoin. Unterschiedliche Handelskalender erzeugen Lücken. Der Goldpreis wird niemals über Nicht-Handelstage fortgeschrieben; exogene Reihen werden höchstens drei Ziel-Handelstage und ausschließlich aus der Vergangenheit fortgeschrieben.

In [ ]:
data_config = load_yaml('data.yaml')
raw_paths = download_all(data_config, force_download=False)
exogenous_names = list(data_config['exogenous'])
data, data_hash = prepare_dataset(raw_paths[0], dict(zip(exogenous_names, raw_paths[1:])))
display(data.describe())
display(Image(filename=str(plot_history(data['gold_usd'], data_hash))))


**Interpretation:** Die Zeitreihe zeigt Preisniveau, Schwankungsbreite und Datenabdeckung. Korrelationen sind deskriptiv und begründen keine Kausalität.

## 3. Datenvorbereitung

Die Hauptanalyse prognostiziert Preisniveaus. Skalierer werden ausschließlich auf Trainingsdaten angepasst. Zielbasierte Lags und rollierende Kennzahlen verwenden nur verschobene Vergangenheitswerte; zentrierte Fenster sind ausgeschlossen. Exogene Variablen können zur historischen Analyse dienen. Für die Zukunft wird bewusst univariat prognostiziert, damit keine unbekannten zukünftigen Informationen stillschweigend verwendet werden.

## 4. Modellierung

Verglichen werden Persistenz, ARIMA, LSTM, N-BEATS und PatchTST. Die neuronalen Modelle sind nativ in PyTorch implementiert, nutzen Early Stopping und stellen den besten Validierungszustand wieder her. `quick_mode` dient der Entwicklung; `full_mode` verwendet die vollständigen Modellgrößen.

In [ ]:
experiment_config = load_yaml('experiments.yaml')
holdout_results = run_holdout(data, cutoff=experiment_config['cutoff_date'], mode=experiment_config['mode'])


## 5. Evaluation

Beim strikt eingefrorenen Ursprung enden Training, Skalierung, Validierung und Modellwahl vor dem 1. Juni 2025. Während der gesamten maskierten Periode werden keine echten Goldpreise als neue Eingaben verwendet. MAE, RMSE, MASE, sMAPE und Richtungsgenauigkeit werden für 1, 7, 30 Tage und den vollständigen Zeitraum berichtet. MASE unter 1 bedeutet eine Verbesserung gegenüber der skalierten naiven Ein-Schritt-Abweichung.

In [ ]:
import pandas as pd
metric_files = list((ROOT / 'artifacts' / 'metrics' / 'holdout').glob('*.csv'))
metrics = pd.concat([pd.read_csv(path) for path in metric_files], ignore_index=True)
display(metrics.sort_values(['horizon', 'mae']))


**Interpretation:** Niedrigere Fehler sind besser; Richtungsgenauigkeit muss im Kontext betrachtet werden. Ein komplexes Modell überzeugt nur, wenn es die Persistenz robust schlägt. Die einmalige Holdout-Periode begrenzt die Verallgemeinerbarkeit und ersetzt kein Backtesting über mehrere Marktregime.

## 6. Bereitstellung

Für die Zukunft werden alle Modelle mit den aktuell verfügbaren Daten neu trainiert. Die Horizonte betragen 7, 30 und 90 Handelstage. Die gespeicherten Artefakte speisen das Streamlit-Dashboard und MLflow. Zukunftsprognosen sind modellbasierte Szenarien mit erheblicher Unsicherheit und ausdrücklich keine Anlageberatung.

In [ ]:
future_forecasts = run_future(data, horizon=90, mode=experiment_config['mode'])
display(future_forecasts.head(30))


**Interpretation:** Unterschiede zwischen Modellen verdeutlichen Modellunsicherheit. Nicht abgebildet sind verlässliche Konfidenzintervalle, Transaktionskosten, strukturelle Brüche und extreme Ereignisse. Entscheidungen dürfen nicht allein auf diesen Punktprognosen beruhen.